1st step: checking data

In [2]:
import pandas as pd

df = pd.read_csv("train.csv")

print(df.shape)
print(df.columns.tolist())

df = df[["comment_text", "toxic"]]

print(df.isna().sum())
print("Duplicates:", df.duplicated().sum())

print(df["toxic"].value_counts())
print(df["toxic"].value_counts(normalize=True))

print("Toxic:")
for t in df[df["toxic"] == 1]["comment_text"].head(3):
    print("-", t[:200])

print("NonToxic:")
for t in df[df["toxic"] == 0]["comment_text"].head(3):
    print("-", t[:200])

(159571, 8)
['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
comment_text    0
toxic           0
dtype: int64
Duplicates: 0
toxic
0    144277
1     15294
Name: count, dtype: int64
toxic
0    0.904156
1    0.095844
Name: proportion, dtype: float64
Toxic:
- COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK
- Hey... what is it..
@ | talk .
What is it... an exclusive group of some WP TALIBANS...who are good at destroying, self-appointed purist who GANG UP any one who asks them questions abt their ANTI-SOCIA
- Bye! 

Don't look, come or think of comming back! Tosser.
NonToxic:
- Explanation
Why the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove th
- D'aww! He matches this background colour I'm seemingly stuck with. Thanks.  (talk) 21:51, January 11, 2016 (UTC)
- Hey man, I'm really not trying to edit war. It's just that 

2nd step: TF-IDF

In [4]:
%pip install scikit-learn joblib

  Using cached scikit_learn-1.9.1-cp313-cp313-macosx_12_0_arm64.whl.metadata (9.3 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached narwhals-2.28.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.1-cp313-cp313-macosx_12_0_arm64.whl (8.3 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached narwhals-2.28.0-py3-none-any.whl (490 kB)
Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl (20.5 MB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: 

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df["comment_text"]
y = df["toxic"]
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(len(X_train), len(X_val), len(X_test))

vectorizer = TfidfVectorizer(max_features=50000)
X_train_tfidf = vectorizer.fit_transform(X_train) 
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

95742 31914 31915


3rd step: Baseline

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_val_tfidf)

p = precision_score(y_val, y_pred)
r = recall_score(y_val, y_pred)
f = f1_score(y_val, y_pred)

print(f"Precision: {p:.4f}")
print(f"Recall:    {r:.4f}")
print(f"F1-score:  {f:.4f}")

results = [{"Model": "Baseline", "Precision": p, "Recall": r, "F1-score": f}]

Precision: 0.9242
Recall:    0.5780
F1-score:  0.7112


4rd step: Experiment with differens sorting methods

In [8]:
def run_experiment(name, C=1.0, class_weight=None, ngram_range=(1, 1)):
    vec = TfidfVectorizer(max_features=50000, ngram_range=ngram_range)
    Xtr = vec.fit_transform(X_train)
    Xva = vec.transform(X_val)

    m = LogisticRegression(C=C, class_weight=class_weight, max_iter=1000)
    m.fit(Xtr, y_train)
    pred = m.predict(Xva)

    row = {
        "Model": name,
        "Precision": precision_score(y_val, pred),
        "Recall": recall_score(y_val, pred),
        "F1-score": f1_score(y_val, pred),
    }
    print(row)
    return row, m, vec

rowA, modelA, vecA = run_experiment("Experiment A (C=10)", C=10)
rowB, modelB, vecB = run_experiment("Experiment B (balanced)", class_weight="balanced")
rowC, modelC, vecC = run_experiment("Experiment C (ngram 1-2)", ngram_range=(1, 2))

results += [rowA, rowB, rowC]

results_df = pd.DataFrame(results).round(4)
print(results_df)
results_df.to_csv("results.csv", index=False)

{'Model': 'Experiment A (C=10)', 'Precision': 0.8769622401357658, 'Recall': 0.6757110166721151, 'F1-score': 0.7632939438700148}
{'Model': 'Experiment B (balanced)', 'Precision': 0.6217765042979942, 'Recall': 0.851258581235698, 'F1-score': 0.7186421967710777}
{'Model': 'Experiment C (ngram 1-2)', 'Precision': 0.926655719759168, 'Recall': 0.5534488394900294, 'F1-score': 0.6930004093327875}
                      Model  Precision  Recall  F1-score  \
0                  Baseline     0.9242  0.5780    0.7112   
1                       NaN     0.8770  0.6757    0.7633   
2                       NaN     0.6218  0.8513    0.7186   
3                       NaN     0.9267  0.5534    0.6930   
4       Experiment A (C=10)     0.8770  0.6757    0.7633   
5   Experiment B (balanced)     0.6218  0.8513    0.7186   
6  Experiment C (ngram 1-2)     0.9267  0.5534    0.6930   

                     Модель  
0                       NaN  
1       Experiment A (C=10)  
2   Experiment B (balanced)  
3  Exper

In [9]:
results = [
    {"Model": "Baseline", "Precision": p, "Recall": r, "F1-score": f},
    rowA, rowB, rowC,
]

results_df = pd.DataFrame(results).round(4)
print(results_df)
results_df.to_csv("results.csv", index=False)

                      Model  Precision  Recall  F1-score
0                  Baseline     0.9242  0.5780    0.7112
1       Experiment A (C=10)     0.8770  0.6757    0.7633
2   Experiment B (balanced)     0.6218  0.8513    0.7186
3  Experiment C (ngram 1-2)     0.9267  0.5534    0.6930


Version A

In [10]:
import joblib

best_model = modelA
best_vec = vecA

X_test_best = best_vec.transform(X_test)
y_test_pred = best_model.predict(X_test_best)

print("TEST Precision:", precision_score(y_test, y_test_pred))
print("TEST Recall:   ", recall_score(y_test, y_test_pred))
print("TEST F1:       ", f1_score(y_test, y_test_pred))

joblib.dump(best_model, "toxic_model.joblib")
joblib.dump(best_vec, "tfidf_vectorizer.joblib")

TEST Precision: 0.8771416631842875
TEST Recall:    0.6861719516181759
TEST F1:        0.7699926632428467


['tfidf_vectorizer.joblib']

In [11]:
def predict_toxic(text):
    vec = best_vec.transform([text])
    pred = best_model.predict(vec)[0]
    prob = best_model.predict_proba(vec)[0][1]
    return ("Toxic" if pred == 1 else "Non-toxic"), round(float(prob), 3)

tests = [
    "Thank you, this article was very helpful!",
    "You are an idiot and nobody likes you",
    "I disagree with your opinion, but I respect it",
]
for t in tests:
    print(t, "->", predict_toxic(t))

Thank you, this article was very helpful! -> ('Non-toxic', 0.0)
You are an idiot and nobody likes you -> ('Toxic', 1.0)
I disagree with your opinion, but I respect it -> ('Non-toxic', 0.002)


In [12]:
from predict import predict_toxic

for t in ["Thank you, this was helpful!", "You are an idiot and nobody likes you", ""]:
    print(repr(t), "->", predict_toxic(t))

'Thank you, this was helpful!' -> ('Non-toxic', 0.0007202641083058031)
'You are an idiot and nobody likes you' -> ('Toxic', 0.9999563959720204)
'' -> ('Empty input', 0.0)
